<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보충 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 처음부터 구현하는 Qwen3 전문가 혼합 모델 (독립형 노트북)

- 이 노트북은 의도적으로 최소한으로 구성되었으며 Qwen3-30B-A3B 모델(**Coder**, **Instruct** 및 **Thinking** 변형 지원) 구현 코드에 중점을 둡니다. 이 모델에 대한 자세한 정보는 원본 블로그 게시물, 기술 보고서 및 모델 허브 페이지를 참조하세요:
  - [Qwen3: Think Deeper, Act Faster](https://qwenlm.github.io/blog/qwen3/)
  - [Qwen3 Technical Report](https://arxiv.org/abs/2505.09388)
  - https://huggingface.co/Qwen/Qwen3-Coder-30B-A3B-Instruct (Qwen3 Coder Flash)
  - https://huggingface.co/Qwen/Qwen3-30B-A3B-Thinking-2507 (새로운 사고 모델)
  - https://huggingface.co/Qwen/Qwen3-235B-A22B-Instruct-2507 (새로운 지시 모델)
  - https://huggingface.co/Qwen/Qwen3-30B-A3B (원본 Instruct/Thinking 하이브리드 모델)
- Qwen3의 많은 아키텍처 구성요소는 Llama 3과 유사합니다. 개별 구성요소와 GPT와 여기서 사용하는 구성요소 간의 관계를 설명하는 단계별 가이드를 원한다면 GPT-to-Llama 변환 노트북을 참조하세요:
  - [처음부터 구현한 GPT 아키텍처를 Llama 2로 변환하기](../07_gpt_to_llama/converting-gpt-to-llama2.ipynb)
  - [처음부터 Llama 2를 Llama 3.2로 변환하기](../07_gpt_to_llama/converting-llama2-to-llama3.ipynb)
  

**기본적으로 이 노트북은 Qwen3-Coder-30B-A3B-Instruct (일명 Qwen3 Coder Flash)를 실행하며 80 GB의 VRAM이 필요합니다 (예: 단일 A100 또는 H100)**

<br>

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/qwen/qwen3-coder-flash-overview.webp?123" width="600px">

<br>
  
- 코드에 대해:
  - 모든 코드는 제 자체 코드로, [Build A Large Language Model (From Scratch)](http://mng.bz/orYv) 책에서 구현한 모델 코드에 Qwen3 아키텍처를 매핑한 것입니다. 코드는 허용적인 오픈소스 Apache 2.0 라이선스 하에 공개됩니다 ([LICENSE.txt](https://github.com/rasbt/LLMs-from-scratch/blob/main/LICENSE.txt) 참조)

In [ ]:
# pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch05/07_gpt_to_llama/requirements-extra.txt

In [ ]:
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # 사전훈련된 가중치를 다운로드하기 위해
    "tokenizers",       # 토크나이저를 구현하기 위해
    "torch",            # 모델을 구현하기 위해
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

&nbsp;
# 1. 아키텍처 코드 (Architecture code)

In [ ]:
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)


class MoEFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_experts_per_tok = cfg["num_experts_per_tok"]
        self.num_experts = cfg["num_experts"]
        self.gate = nn.Linear(cfg["emb_dim"], cfg["num_experts"], bias=False, dtype=cfg["dtype"])

        # 가중치 로딩 전 모델 초기화 시 메모리 압박을 줄이기 위한 meta 디바이스
        meta_device = torch.device("meta")
        self.fc1 = nn.ModuleList([
            nn.Linear(
                cfg["emb_dim"], cfg["moe_intermediate_size"],
                bias=False, dtype=cfg["dtype"], device=meta_device)
            for _ in range(cfg["num_experts"])]
        )
        self.fc2 = nn.ModuleList([
            nn.Linear(
                cfg["emb_dim"], cfg["moe_intermediate_size"],
                bias=False, dtype=cfg["dtype"], device=meta_device
                )
            for _ in range(cfg["num_experts"])]
        )
        self.fc3 = nn.ModuleList([
            nn.Linear(
                cfg["moe_intermediate_size"], cfg["emb_dim"],
                bias=False, dtype=cfg["dtype"], device=meta_device
                )
            for _ in range(cfg["num_experts"])]
        )

    def forward(self, x):
        b, seq_len, embed_dim = x.shape
        scores = self.gate(x)  # (b, seq_len, num_experts)
        topk_scores, topk_indices = torch.topk(scores, self.num_experts_per_tok, dim=-1)
        topk_probs = torch.softmax(topk_scores, dim=-1)
        
        expert_outputs = []
        for e in range(self.num_experts):
            hidden = torch.nn.functional.silu(self.fc1[e](x)) * self.fc2[e](x)
            out = self.fc3[e](hidden)
            expert_outputs.append(out.unsqueeze(-2))
        expert_outputs = torch.cat(expert_outputs, dim=-2)  # (b, t, num_experts, emb_dim)

        gating_probs = torch.zeros_like(scores)

        for i in range(self.num_experts_per_tok):
            indices = topk_indices[..., i:i+1]
            prob = topk_probs[..., i:i+1]
            gating_probs.scatter_(dim=-1, index=indices, src=prob)
        gating_probs = gating_probs.unsqueeze(-1)  # (b, t, num_experts, 1)
        
        # 전문가들에 대한 가중 합계
        y = (gating_probs * expert_outputs).sum(dim=-2)
        return y


        # 어떤 이유로, 아래 버전은 사용되지 않는 전문가도 계산하는
        # 위의 단순한 버전보다 느립니다

        # def forward(self, x):
        #     scores = self.gate(x)  # (b, seq_len, num_experts)
        #     topk_scores, topk_indices = torch.topk(scores, self.num_experts_per_tok, dim=-1)
        #     topk_probs = torch.softmax(topk_scores, dim=-1)
        #     y = torch.zeros_like(x)
        #
        #     for i in range(self.num_experts_per_tok):
        #         # expert_indices는 [0, num_experts) 범위의 값을 가진 (b, seq_len)
        #         expert_indices = topk_indices[..., i]
        #         prob = topk_probs[..., i].unsqueeze(-1)  # (b, seq_len, 1)
        #
        #         # 각 전문가에 대해, 할당된 토큰만 처리
        #         for e in range(self.num_experts):
        #             mask = (expert_indices == e)  # (b, seq_len) 불린 마스크
        #             if mask.any():
        #                 selected = x[mask]  # (num_tokens_e, emb_dim)
        #                 out = self.fc3[e](torch.nn.functional.silu(self.fc1[e](selected)) * self.fc2[e](selected))
        #                 y[mask] += prob[mask] * out
        #     return y

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale

        if self.shift is not None:
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)

In [ ]:
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "임베딩 차원은 짝수여야 합니다"

    # 역주파수 계산
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # 위치 인덱스 생성
    positions = torch.arange(context_length, dtype=dtype)

    # 각도 계산
    angles = positions[:, None] * inv_freq[None, :]  # Shape: (context_length, head_dim // 2)

    # head_dim에 맞게 각도 확장
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # 사인과 코사인 사전 계산
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin, offset=0):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "헤드 차원은 짝수여야 합니다"

    # x를 앞쪽 절반과 뒤쪽 절반으로 분할
    x1 = x[..., : head_dim // 2]  # 앞쪽 절반
    x2 = x[..., head_dim // 2:]  # 뒤쪽 절반

    # sin과 cos 형태 조정
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    # 회전 변환 적용
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # cos와 sin 회전을 적용한 후 낮은 정밀도 사용 가능
    return x_rotated.to(dtype=x.dtype)

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads는 num_kv_groups로 나누어떨어져야 합니다"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`head_dim`이 설정되지 않은 경우 `d_in`은 `num_heads`로 나누어떨어져야 합니다"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape

        # 프로젝션 적용
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # 재구성
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys_new = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # 선택적 정규화
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys_new = self.k_norm(keys_new)

        # RoPE 적용
        queries = apply_rope(queries, cos, sin, offset=start_pos)
        keys_new = apply_rope(keys_new, cos, sin, offset=start_pos)

        if cache is not None:
            prev_k, prev_v = cache
            keys = torch.cat([prev_k, keys_new], dim=2)
            values = torch.cat([prev_v, values_new], dim=2)
            next_cache = (keys, values)
        else:
            start_pos = 0  # RoPE 재설정
            keys, values = keys_new, values_new
            next_cache = (keys, values)

        # K와 V를 헤드 수에 맞게 확장
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # 어텐션
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)

        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context), next_cache

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        if cfg["num_experts"] > 0:
            self.ff = MoEFeedForward(cfg)
        else:
            self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        # 어텐션 블록에 대한 단축 연결
        shortcut = x
        x = self.norm1(x)
        x, next_cache = self.att(x, mask, cos, sin, start_pos=start_pos, cache=cache)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # 원래 입력을 다시 추가

        # 피드포워드 블록에 대한 단축 연결
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # 원래 입력을 다시 추가

        return x, next_cache

In [ ]:
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # 주요 모델 파라미터
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(  # Sequential은 하나의 입력만 받을 수 있고, 우리는 `x, mask, cos, sin`이 필요하므로 ModuleList 사용
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # 재사용 가능한 유틸리티
        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg
        self.current_pos = 0  # KV 캐시에서 현재 위치 추적


    def forward(self, in_idx, cache=None):
        # 순전파
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + num_tokens
            self.current_pos = pos_end
            mask = torch.triu(
                torch.ones(pos_end, pos_end, device=x.device, dtype=torch.bool), diagonal=1
            )[pos_start:pos_end, :pos_end]
        else:
            pos_start = 0  # 반드시 필요하지 않지만 torch.compile에 도움
            mask = torch.triu(
                torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1
            )
        # 배치와 헤드에 브로드캐스트하기 위한 형태 (1, 1, num_tokens, num_tokens)
        mask = mask[None, None, :, :]

        next_cache = []
        for i, block in enumerate(self.trf_blocks):
            blk_cache = cache.get(i) if cache else None
            x, new_blk_cache = block(x, mask, self.cos, self.sin,
                                     start_pos=pos_start,
                                     cache=blk_cache)
            if cache is not None:
                cache.update(i, new_blk_cache)
            next_cache.append(new_blk_cache)

        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

    def reset_kv_cache(self):
        self.current_pos = 0

In [ ]:
class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None

&nbsp;
# 2. 모델 초기화 (Initialize model)

In [ ]:
# 다음에 동일한 구성 사용:

# https://huggingface.co/Qwen/Qwen3-Coder-30B-A3B-Instruct (Qwen3 Coder Flash)
# https://huggingface.co/Qwen/Qwen3-30B-A3B-Thinking-2507
# https://huggingface.co/Qwen/Qwen3-235B-A22B-Instruct-2507
# https://huggingface.co/Qwen/Qwen3-30B-A3B (원본 Instruct/Thinking 하이브리드 모델)

QWEN3_CONFIG = {
    "vocab_size": 151_936,
    "context_length": 262_144,
    "emb_dim": 2048,
    "n_heads": 32,
    "n_layers": 48,
    "head_dim": 128,
    "qk_norm": True,
    "n_kv_groups": 4,
    "rope_base": 10_000_000.0,
    "dtype": torch.bfloat16,
    "num_experts": 128,
    "num_experts_per_tok": 8,
        "moe_intermediate_size": 768,
}

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

In [ ]:
torch.manual_seed(123)

with device:
    model = Qwen3Model(QWEN3_CONFIG)

#model.to(device)

- 계속하기 전에 순전파가 작동하는지 빠르게 확인합니다 (메모리를 절약하기 위해 인스턴스화 시 "meta" 디바이스를 사용하므로 지금은 nan 값이 나와도 괜찮습니다):

In [ ]:
model(torch.tensor([1, 2, 3]).unsqueeze(0).to(device))

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"총 파라미터 수: {total_params:,}")

# 가중치 공유 고려
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\n총 고유 파라미터 수: {total_params_normalized:,}")

In [ ]:
def model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # 파라미터당 총 원소 수 계산
        param_size = param.numel()
        total_params += param_size
        # 이 파라미터에 대한 그래디언트가 저장되는지 확인
        if param.requires_grad:
            total_grads += param_size

    # 버퍼 크기 계산 (메모리를 요구하는 파라미터가 아닌 것들)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # 바이트 단위 크기 = (원소 수) * (각 원소의 바이트 크기)
    # 파라미터와 그래디언트가 입력 dtype과 동일한 타입으로 저장된다고 가정
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # 바이트를 기가바이트로 변환
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch 기본값): {model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

- 걱정하지 마세요. 일부 레이어를 CPU RAM으로 오프로드하여 80 GB RAM을 가진 A100 카드에서 모델이 잘 실행됩니다

&nbsp;
# 4. 사전훈련된 가중치 로딩 (Load pretrained weights)

In [ ]:
def load_weights_into_qwen(model, param_config, params):
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"텐서 '{tensor_name}'에서 형태 불일치. 왼쪽: {left.shape}, 오른쪽: {right.shape}")
        return torch.nn.Parameter(right.clone().detach() if isinstance(right, torch.Tensor) else torch.tensor(right))

    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V 프로젝션
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )

        # 출력 프로젝션
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )

        # QK 정규화
        if hasattr(att, "q_norm") and att.q_norm is not None:
            att.q_norm.scale = assign(
                att.q_norm.scale,
                params[f"model.layers.{l}.self_attn.q_norm.weight"],
                f"model.layers.{l}.self_attn.q_norm.weight"
            )
        if hasattr(att, "k_norm") and att.k_norm is not None:
            att.k_norm.scale = assign(
                att.k_norm.scale,
                params[f"model.layers.{l}.self_attn.k_norm.weight"],
                f"model.layers.{l}.self_attn.k_norm.weight"
            )

        # 어텐션 레이어노름
        block.norm1.scale = assign(
            block.norm1.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # 피드포워드 가중치
        if "num_experts" in param_config:
            # 라우터(게이팅) 가중치 로딩
            block.ff.gate.weight = assign(
                block.ff.gate.weight,
                params[f"model.layers.{l}.mlp.gate.weight"],
                f"model.layers.{l}.mlp.gate.weight"
            )
            # 전문가 가중치 로딩
            for e in range(param_config["num_experts"]):
                prefix = f"model.layers.{l}.mlp.experts.{e}"
                block.ff.fc1[e].weight = assign(
                    block.ff.fc1[e].weight,
                    params[f"{prefix}.gate_proj.weight"],
                    f"{prefix}.gate_proj.weight"
                )
                block.ff.fc2[e].weight = assign(
                    block.ff.fc2[e].weight,
                    params[f"{prefix}.up_proj.weight"],
                    f"{prefix}.up_proj.weight"
                )
                block.ff.fc3[e].weight = assign(
                    block.ff.fc3[e].weight,
                    params[f"{prefix}.down_proj.weight"],
                    f"{prefix}.down_proj.weight"
                )
                # 가중치 할당 후, 전문가 레이어를 meta에서 CPU로 이동
                block.ff.fc1[e] = block.ff.fc1[e].to("cpu")
                block.ff.fc2[e] = block.ff.fc2[e].to("cpu")
                block.ff.fc3[e] = block.ff.fc3[e].to("cpu")

        else:
            block.ff.fc1.weight = assign(
                block.ff.fc1.weight,
                params[f"model.layers.{l}.mlp.gate_proj.weight"],
                f"model.layers.{l}.mlp.gate_proj.weight"
            )
            block.ff.fc2.weight = assign(
                block.ff.fc2.weight,
                params[f"model.layers.{l}.mlp.up_proj.weight"],
                f"model.layers.{l}.mlp.up_proj.weight"
            )
            block.ff.fc3.weight = assign(
                block.ff.fc3.weight,
                params[f"model.layers.{l}.mlp.down_proj.weight"],
                f"model.layers.{l}.mlp.down_proj.weight"
            )

        block.norm2.scale = assign(
            block.norm2.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # 최종 정규화와 출력 헤드
    model.final_norm.scale = assign(model.final_norm.scale, params["model.norm.weight"], "model.norm.weight")

    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        # 모델이 가중치 공유를 사용하므로 여기서 임베딩 레이어 가중치를 재사용
        print("모델이 가중치 공유를 사용합니다.")
        model.out_head.weight = assign(model.out_head.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

In [ ]:
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import snapshot_download

repo_id = "Qwen/Qwen3-30B-A3B"  # 원본 Instruct/Thinking 하이브리드 모델
repo_id = "Qwen/Qwen3-235B-A22B-Instruct-2507"  # 새로운 지시 모델
repo_id = "Qwen/Qwen3-30B-A3B-Thinking-2507"  # 새로운 사고 모델
repo_id = "Qwen/Qwen3-Coder-30B-A3B-Instruct"  # (Qwen3 Coder Flash)

local_dir = Path(repo_id).parts[-1]

repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
index_path = os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path, "r") as f:
    index = json.load(f)

weights_dict = {}
for filename in set(index["weight_map"].values()):
    shard_path = os.path.join(repo_dir, filename)
    shard = load_file(shard_path)
    weights_dict.update(shard)

load_weights_into_qwen(model, QWEN3_CONFIG, weights_dict)
model.to(device);

&nbsp;
# 4. 토크나이저 로딩 (Load tokenizer)

In [ ]:
import re
from tokenizers import Tokenizer


class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>)")

    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {t: self._tok.token_to_id(t) for t in self._SPECIALS}

        self.pad_token_id = self._special_to_id.get("<|endoftext|>")
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s

In [ ]:
tokenizer_file_path = f"{Path(repo_id).parts[-1]}/tokenizer.json"

tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_file_path,
    repo_id=repo_id,
    add_generation_prompt=True,
    add_thinking=True
)

In [ ]:
# prompt = "대형 언어 모델에 대한 간단한 소개를 해주세요."
prompt = "Python에서 이진 탐색 함수를 구현하세요"


input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

&nbsp;
# 5. 텍스트 생성 (Generate text)

In [ ]:
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None, context_size=None):
    model.eval()

    with torch.no_grad():
        cache = KVCache(n_layers=model.cfg["n_layers"])
        model.reset_kv_cache()

        # 초기 컨텍스트로 캐시를 준비
        logits = model(token_ids, cache=cache)

        for _ in range(max_new_tokens):
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)

            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)

            # 모델에 새로운 토큰만 공급; 캐시가 히스토리 처리
            logits = model(next_token, cache=cache)

In [ ]:
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=200,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

&nbsp;
# 다음 단계는? (What's next?)

- `llms_from_scratch` 패키지를 통해 이 모델을 사용하려면 [README.md](./README.md)를 확인하세요
- 처음부터 대형 언어 모델을 구축하고 그 메커니즘을 더 깊이 이해하는 포괄적인 가이드에 관심이 있다면 [Build a Large Language Model (From Scratch)](http://mng.bz/orYv) 책을 좋아하실 것입니다

<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>